# Dataset, DataLoader & Augmentations

Use the three companion topic pages for the detailed explanations. Run the CPU setup and prepared data/RLE helpers, then inspect Dataset, batching and the paired augmentation code. Images are original synthetic cloud-like examples.


In [ ]:
%pip install numpy pandas pillow torch matplotlib


## 1. Imports and the teaching contract

A Dataset answers “what is sample i?” A DataLoader answers “which samples go together?” We will return one RGB image and four binary masks per sample. No model or GPU is needed.

`Path` builds portable file paths. NumPy handles pixels, pandas reads the CSV, Pillow opens images, and PyTorch builds tensors. The class order below is a contract: mask channel 0 always means Fish, even when Fish is absent. These names describe cloud patterns, not objects in the photograph.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader, Subset
import matplotlib.pyplot as plt

CLASSES = ("Fish", "Flower", "Gravel", "Sugar")
ROOT = Path("runs/cloud_loader_demo")
ROOT.mkdir(parents=True, exist_ok=True)



## 2. RLE: the CSV stores coordinates compactly

A mask is an H × W array of zeros and ones. RLE stores foreground runs as start–length pairs. Starts are 1-based; Python slices are 0-based and exclude the end. `order="F"` traverses down each column.

In the encoder, zero padding makes every foreground run have a beginning and end. `np.diff` finds transitions; subtracting alternating transitions produces lengths. In the decoder, an empty field produces an all-zero mask. We reject malformed runs rather than silently teaching students that invalid annotations are valid. This parser assumes sorted, non-overlapping runs.


In [ ]:
def encode_rle(mask):
    pixels = np.asarray(mask, dtype=np.uint8).reshape(-1, order="F")
    padded = np.concatenate(([0], pixels, [0]))
    changes = np.flatnonzero(np.diff(padded)) + 1
    changes[1::2] -= changes[::2]
    return " ".join(map(str, changes))


def decode_rle(text, height, width):
    flat = np.zeros(height * width, dtype=np.uint8)
    if pd.isna(text) or not str(text).strip():
        return flat.reshape(height, width, order="F")
    values = np.array(str(text).split(), dtype=np.int64)
    if len(values) % 2:
        raise ValueError("RLE needs start-length pairs")
    previous_end = 0
    for start, length in values.reshape(-1, 2):
        lo = int(start) - 1
        hi = lo + int(length)
        if lo < previous_end or length <= 0 or hi > len(flat):
            raise ValueError("Invalid or overlapping RLE run")
        flat[lo:hi] = 1
        previous_end = hi
    return flat.reshape(height, width, order="F")

small = np.zeros((4, 6), dtype=np.uint8)
small[1:3, 2] = 1
assert encode_rle(small) == "10 2"
assert np.array_equal(decode_rle("10 2", 4, 6), small)
assert decode_rle("", 4, 6).sum() == 0
print("RLE round trip passed")



## 3. Create a small, independent dataset

These pictures are generated locally. They are not Kaggle images and do not reproduce competition examples. The layout imitates the cloud dataset: `train_images/`, `test_images/`, `train.csv`, and `sample_submission.csv`. Training has one row per image–class pair with `Image_Label` and `EncodedPixels`.

We use 60 × 90 RGB images to keep the lesson fast. Toy labels are ellipses, not scientifically meaningful cloud annotations. Some images have no cloud labels, others have multiple labels, and masks may overlap. Every image still has four CSV rows. Test images have no training labels. The sample submission is only a structural template, not a valid entry to any real competition.


In [ ]:
def create_demo(root, n_train=24, n_test=4):
    rng = np.random.default_rng(42)
    height, width = 60, 90
    yy, xx = np.mgrid[:height, :width]
    rows, submission = [], []
    for folder, count in (("train_images", n_train), ("test_images", n_test)):
        (root / folder).mkdir(exist_ok=True)
        for i in range(count):
            filename = f"demo_{i:03d}.jpg"
            image = np.zeros((height, width, 3), dtype=np.float32)
            image[:] = (35, 90, 135)
            image += rng.normal(0, 5, image.shape)
            for channel, label in enumerate(CLASSES):
                mask = np.zeros((height, width), dtype=np.uint8)
                if i % 6 != 0 and (i + channel) % 3 != 0:
                    cx, cy = rng.integers(18, 72), rng.integers(15, 45)
                    mask = (((xx-cx)/16)**2 + ((yy-cy)/9)**2 < 1).astype(np.uint8)
                    image[mask.astype(bool)] += 25 + channel * 8
                key = f"{filename}_{label}"
                if folder == "train_images":
                    rows.append({"Image_Label": key, "EncodedPixels": encode_rle(mask)})
                else:
                    submission.append({"Image_Label": key, "EncodedPixels": ""})
            Image.fromarray(np.clip(image, 0, 255).astype(np.uint8)).save(root / folder / filename)
    pd.DataFrame(rows).to_csv(root / "train.csv", index=False)
    pd.DataFrame(submission).to_csv(root / "sample_submission.csv", index=False)

create_demo(ROOT)
table = pd.read_csv(ROOT / "train.csv", keep_default_na=False)
print(table.head(8).to_string(index=False))
print("CSV rows:", len(table), "— training images:", len(table) // len(CLASSES))



## 4. Build an index: one image, four rows

`rsplit("_", n=1)` splits from the right: `demo_001.jpg_Fish` becomes `demo_001.jpg` and `Fish`, preserving underscores in filenames. Reading with `keep_default_na=False` retains empty mask strings.

The CSV contains image–class rows; our Dataset indexes images. `pivot` creates one row per filename and one column per class. `reindex` locks channel order. We validate missing pairs before reshaping: a missing annotation row is different from an explicitly empty mask. Duplicate image–class rows should also fail rather than be silently combined.


In [ ]:
def read_index(root):
    frame = pd.read_csv(root / "train.csv", keep_default_na=False)
    parts = frame["Image_Label"].str.rsplit("_", n=1, expand=True)
    frame["filename"], frame["label"] = parts[0], parts[1]
    if not frame["label"].isin(CLASSES).all():
        raise ValueError("Unknown class")
    if frame.duplicated(["filename", "label"]).any():
        raise ValueError("Duplicate image-class pair")
    counts = frame.groupby("filename")["label"].nunique()
    if not counts.eq(len(CLASSES)).all():
        raise ValueError("Every image needs four explicit annotation rows")
    return frame.pivot(index="filename", columns="label", values="EncodedPixels").reindex(columns=CLASSES)

index = read_index(ROOT)
print(index.index[:3].tolist())
print("Dataset samples:", len(index), "Mask channels:", len(index.columns))



## 5. Write the Dataset: constructor, length, item

`__init__` stores paths and builds a small annotation index once. It does not load every image into RAM. `__len__` counts images. `__getitem__(i)` opens one image on demand, decodes its four masks, and returns a dictionary.

Pillow gives RGB `[H,W,3]` bytes. Dividing by 255 converts intensities to `[0,1]`; this is scaling, not mean/std standardization. `permute(2,0,1)` changes axis order to `[3,H,W]`. `np.stack` puts four masks into `[4,H,W]`. Floating masks are convenient for a future BCE loss. The filename is metadata, not a model input.

Decode at the original image dimensions before resizing. Images use bilinear interpolation, while binary masks use nearest-neighbor to preserve 0/1 labels. `size` is `(height,width)`, but Pillow expects `(width,height)`. The toy defaults preserve its 3:2 aspect ratio. A single sample has no batch axis yet.


In [ ]:
class CloudDataset(Dataset):
    def __init__(self, root, size=(60, 90), transform=None):
        self.root = Path(root)
        self.index = read_index(self.root)
        self.size = size
        self.transform = transform

    def __len__(self):
        return len(self.index)

    def __getitem__(self, i):
        filename = self.index.index[i]
        with Image.open(self.root / "train_images" / filename) as source:
            image = source.convert("RGB")
        width, height = image.size
        masks = [decode_rle(self.index.iloc[i][label], height, width) for label in CLASSES]
        out_h, out_w = self.size
        image = image.resize((out_w, out_h), Image.Resampling.BILINEAR)
        masks = [np.asarray(Image.fromarray(m).resize(
            (out_w, out_h), Image.Resampling.NEAREST), dtype=np.float32) for m in masks]
        image_array = np.asarray(image, dtype=np.float32) / 255.0
        image_tensor = torch.from_numpy(image_array).permute(2, 0, 1).contiguous()
        mask_tensor = torch.from_numpy(np.stack(masks))
        if self.transform is not None:
            image_tensor, mask_tensor = self.transform(image_tensor, mask_tensor)
        return {"image": image_tensor, "mask": mask_tensor, "filename": filename}

dataset = CloudDataset(ROOT)
sample = dataset[1]
print("Images:", len(dataset))
print(sample["filename"], sample["image"].shape, sample["mask"].shape)
assert sample["image"].dtype == torch.float32
assert sample["mask"].shape == (4, 60, 90)
assert set(sample["mask"].unique().tolist()) <= {0.0, 1.0}



## 6. Look before you batch

Correct shapes do not prove correct alignment. Display the image beside each mask and inspect an overlay. Matplotlib wants `[H,W,C]`, so we permute the image back for display. The masks stay two-dimensional. The masked array hides zeros so the image remains visible below the foreground.


In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
rgb = sample["image"].permute(1, 2, 0).numpy()
axes[0].imshow(rgb)
axes[0].set_title(sample["filename"])
for channel, label in enumerate(CLASSES):
    axes[channel + 1].imshow(rgb)
    visible = np.ma.masked_where(sample["mask"][channel].numpy() == 0,
                                 sample["mask"][channel].numpy())
    axes[channel + 1].imshow(visible, cmap="autumn", vmin=0, vmax=1, alpha=0.6)
    axes[channel + 1].set_title(label)
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()



## 7. Split images, then create DataLoaders

Splitting CSV rows can put the same photograph into training and validation under different labels. Here we split the already indexed images. This tiny random holdout demonstrates isolation; for real rare-label data, audit class support and use multi-label stratification or grouping when appropriate.

`Subset` translates its local index into the original Dataset index. The seeded generator makes this split repeatable. It is separate from the generator that controls training shuffling. `shuffle=True` changes training sample order, not pixels or labels. Validation uses a fixed order. `num_workers=0` keeps loading in the notebook process and gives clear errors.

The default collator stacks matching dictionary tensor values and collects filenames. Four `[3,60,90]` images become `[4,3,60,90]`; masks become `[4,4,60,90]`. `drop_last=False` keeps a smaller final batch. DataLoader iteration creates an iterator; `next` requests its first batch.


# The 80/20 split: practice on some images, check on others

Before making batches, decide which images the model may learn from and which images you will use to check its progress. This is the **train–validation split**.

## Why do we need a split?

Imagine preparing for an exam. If you practise ten questions and then test yourself on those same ten questions, a high score may show that you remember the answers. It does not tell you how well you will answer a new question.

A segmentation model can do something similar: become very good at the training images without learning patterns that work on other images. We call this **overfitting**. To check whether learning transfers to images outside training, we keep some labelled images aside.

- **Training set:** images and masks used to calculate training loss and update model weights.
- **Validation set:** different images and masks used to measure progress and compare choices. Validation masks let us check the predictions, but are not used for gradient updates.

We want the model to recognize cloud patterns in a new image, not merely remember where a cloud appeared in an old image.

## What does 80/20 mean?

With 100 labelled images, an 80/20 split puts 80 into training and 20 into validation. Each image goes into one group, together with all its labels.

Our small demonstration has **24 labelled training images**. We calculate:

```python
n_train = int(0.8 * 24)  # 19
n_val = 24 - n_train    # 5
```

`0.8 * 24` is 19.2. `int` rounds this positive number down to 19, and the five remaining images go to validation. So our actual proportions are approximately 79.2% and 20.8%. Whole images cannot be divided into fractional samples.

80/20 is a useful starting choice, not a rule that is always best. More training images give the model more examples to learn from. More validation images usually make our estimate less sensitive to a few unusual examples. With just five validation images, a result can change a lot because of one difficult image; this toy split teaches mechanics, not a reliable performance estimate.

## Training, validation and test are different jobs

The folder named `train_images/` contains the labelled images available to us. We split those into our own training and validation groups. We do not need to move files into new folders.

The separate `test_images/` folder in our demo has no training labels. It is not the 20% validation group. In a competition, test labels are commonly hidden from participants.

During training, we may look at validation results to choose an epoch, augmentation strength or model setting. Those choices use information from validation, so validation is not an untouched final test. A separate test set, when available with suitable evaluation, checks the final chosen approach.

## Split images, not CSV rows

Recall that one image has four annotation rows. For example:

```text
demo_001.jpg_Fish
demo_001.jpg_Flower
demo_001.jpg_Gravel
demo_001.jpg_Sugar
```

If we split these rows independently, Fish might go into training while Flower goes into validation. Both groups would still contain the same photograph. A model that has already seen that image is not being checked on an unseen image. This is **data leakage**: information crosses the boundary we intended to keep.

Our Dataset already groups all four labels into one image sample. Splitting its image indices keeps all four mask channels together. If multiple images are near-duplicates, adjacent frames, or crops of the same original, split by their shared source group as well; different filenames alone do not guarantee independence.


### What each statement does

`len(dataset)` asks the Dataset's `__len__` method for its number of image samples.

`torch.Generator()` creates a random-number generator. `manual_seed(42)` sets its starting state. The number 42 is simply a chosen seed; it does not make a split more accurate or guarantee balanced classes.

`torch.randperm(n_images)` creates a permutation of the integers from 0 to `n_images - 1`. Each index appears once. The generator makes the ordering reproducible for the same setup and Dataset index order. `.tolist()` converts the tensor into an ordinary Python list.

`order[:cut]` selects positions before `cut`. `order[cut:]` selects positions from `cut` to the end. These are two slices of the same permutation, so they cannot share an index.

`set(...)` lets us compare membership without caring about order. `isdisjoint` checks that the two groups share nothing. The `|` symbol makes their union; comparing it with `set(range(n_images))` checks that no image was left out. The first assertion catches datasets too small for this particular split rule.

`Subset(dataset, train_indices)` stores a connection to the Dataset and a list of allowed indices. It does not copy or load all images. For example, `train_data[0]` asks the original Dataset for whichever image index appears first in `train_indices`.

## Keep the split fixed when comparing experiments

Suppose experiment A and experiment B use different validation images. A better score for B could mean B is better, or simply that its validation images are easier. Keep the same split when comparing changes.

The seed reproduces membership only while the input image list and ordering also stay the same. Save the train and validation filenames for a lasting experiment record, especially if the dataset may change. Do not keep changing seeds until you find a flattering validation score.

## Random does not mean balanced

Imagine a class appears in only two images. A random split can put both into training, leaving none of that class in validation. We then cannot assess performance on positive examples of that class.

After splitting, count how many images contain each label in each group. **Stratification** tries to preserve label representation. In a multi-label task, an image can contain several labels; multi-label stratification considers those patterns rather than treating each annotation row as a separate sample. Very rare label combinations may be too small to distribute proportionally, so inspect the resulting split rather than assuming a method guarantees coverage.

For today's toy lab, use the simple seeded split to understand the mechanism. Before trusting real validation results, audit class coverage and any shared source groups.


See the split topic page for the full code and activities.


In [ ]:
order = torch.randperm(len(dataset), generator=torch.Generator().manual_seed(42)).tolist()
cut = int(0.8 * len(order))
train_indices, val_indices = order[:cut], order[cut:]
assert set(train_indices).isdisjoint(val_indices)
train_data = Subset(dataset, train_indices)
val_data = Subset(dataset, val_indices)
train_loader = DataLoader(
    train_data, batch_size=4, shuffle=True, num_workers=0,
    drop_last=False, generator=torch.Generator().manual_seed(123),
)
val_loader = DataLoader(val_data, batch_size=4, shuffle=False, num_workers=0)
batch = next(iter(train_loader))
print(batch["image"].shape, batch["mask"].shape)
print(batch["filename"])
print("Training images:", len(train_data), "Training batches:", len(train_loader))
print("Validation images:", len(val_data), "Validation batches:", len(val_loader))



## 8. Trace one epoch and check edge cases

An epoch visits all samples once when using this ordinary shuffled loader. A batch is a group of samples; its size can differ from the number of mask channels. With 19 training images and batch size 4, the last batch has 3 images.

The loop below only inspects data. A future training loop would send both tensors to the device and call a model, loss, backward, and optimizer step. DataLoader does none of those things. An empty mask is a valid target, not an unreadable image.


In [ ]:
seen = []
for batch_number, batch in enumerate(train_loader, start=1):
    images, masks = batch["image"], batch["mask"]
    assert images.shape[0] == masks.shape[0]
    assert images.shape[-2:] == masks.shape[-2:]
    seen.extend(batch["filename"])
    print("Batch", batch_number, "contains", images.shape[0], "images")
assert len(seen) == len(set(seen)) == len(train_data)
assert dataset[0]["mask"].sum().item() == 0
print("One complete epoch and clean-image check passed")



## Paired augmentations

Sample each geometric decision once for both tensors; change brightness only on the image. Use separate Dataset objects for training and validation.


In [ ]:
def augment_pair(image, mask):
    if torch.rand(()).item() < 0.5:
        image = torch.flip(image, dims=(-1,))
        mask = torch.flip(mask, dims=(-1,))
    if torch.rand(()).item() < 0.5:
        image = torch.flip(image, dims=(-2,))
        mask = torch.flip(mask, dims=(-2,))
    brightness = 0.8 + 0.4 * torch.rand(()).item()
    image = (image * brightness).clamp(0, 1)
    return image, mask

# Separate Dataset objects keep validation free of random augmentation.
torch.manual_seed(42)
augmented_dataset = CloudDataset(ROOT, transform=augment_pair)
training = Subset(augmented_dataset, train_indices)
validation = Subset(CloudDataset(ROOT), val_indices)
augmented_loader = DataLoader(training, batch_size=4, shuffle=True, num_workers=0,
                             generator=torch.Generator().manual_seed(123))
augmented_batch = next(iter(augmented_loader))
assert set(augmented_batch["mask"].unique().tolist()) <= {0.0, 1.0}
assert torch.equal(validation[0]["image"], validation[0]["image"])
print("Augmented batch:", augmented_batch["image"].shape)

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax in axes:
    view = augmented_dataset[1]
    ax.imshow(view["image"].permute(1, 2, 0))
    foreground = view["mask"].any(dim=0).numpy()
    ax.contour(foreground, levels=[0.5], colors=["yellow"], linewidths=1)
    ax.set_title("Same filename, new transform")
    ax.axis("off")
plt.tight_layout()
plt.show()


## Three-topic review

Explain how Dataset prepares one sample, how DataLoader stacks samples, and why geometric augmentation must share one random decision between image and masks. Explore the companion browser augmentation labs before revealing each activity answer.
